# Exercise 03: Medical Imaging and AI
**AI in Medical Robotics, WS 2026/27.** Lecture 03: Medical Imaging and AI.

In this exercise you test two AI models on medical images. The first model reads chest X-rays of children and answers: pneumonia or normal? The second model draws the outline of a polyp in colonoscopy images. Both models look very good at first. Your job is to find out where they make mistakes, and which mistakes matter.

**At home, before the session:** work through parts 1.0 and 2.0, from top to bottom. The session needs them, so please do them before you come.
1. Run the cells one after the other: click the play button on the left of a cell (or press Shift+Enter).
2. Grey cells that show only a title, for example "Setup: run this cell first", only need to be run. You do not have to read their code.
3. In a few cells you write short lines of code yourself. They are marked `YOUR CODE (home)`. Above each one you find a task, an example and hints. Click a hint to open it.
4. Right below your code there is a check cell. It tells you if your code works. If it does not work yet, the notebook uses a reference version, so you never get stuck.
5. Write your predictions and short answers on the exercise sheet, under the same numbers.
6. The parts called "In class" are for the session. At home you may skip them, or run them after you have written your predictions on the sheet.

**In class:**
1. Open the notebook again and run it from the top: *Runtime, Run all*.
2. Change only the values marked `EDIT`. Each "In class" part tells you which cell to use, what to change and which line of the output to read.
3. If something did not work at home, that is no problem: after *Runtime, Run all* the home cells use the reference versions, and you can take part in everything.

**One number for each part.** A part has the same number here, on the sheet and on the slides. Question 1.2 on a slide is section 1.2 in this notebook and item 1.2 on the sheet. Parts ending in .0 are the homework. To jump to a number, open the table of contents in Colab's left sidebar and click the number.

| Task | At home, before the session | In class |
|---|---|---|
| 1 Classification | 1.0 | 1.1, 1.2, 1.3 |
| 2 Segmentation | 2.0 | 2.1, 2.2, 2.3 |


*Runtime:* choose a GPU first: *Runtime, Change runtime type, T4 GPU*. The notebook also runs without a GPU, but then the training in Task 2 takes about 12 minutes instead of one.

---
# Task 1: Image classification, which error costs more?
*Lecture slides 38, 40, 46, 65 and 66.*

**What is pneumonia?** Pneumonia is an infection of the lungs. The small air sacs in the lung fill with fluid or pus, so the child has fever, a cough and trouble breathing. It is a common reason why young children are taken to hospital. On a chest X-ray, healthy lungs look dark, because air lets the X-rays through. With pneumonia, parts of the lung look white and cloudy.

**The situation.** A hospital wants an AI that reads children's chest X-rays and flags pneumonia (marks the X-ray as "pneumonia"), so that sick children are treated quickly. The company that sells the AI (the vendor) reports 86% accuracy: 86 of 100 X-rays get the right answer. That sounds good. But what does it mean for a sick child who is sent home? What does it mean for a healthy child who gets antibiotics without need? And does the model still work in another hospital?

**What you do in Task 1.** At home (1.0) you write the function that counts the right and wrong answers of the model. In class you see how a model copies the mistakes in its training labels (1.1). You choose a decision threshold for two very different uses in the clinic (1.2). And you catch a model that learned something it should not have learned (1.3).

In [ ]:
#@title Setup: run this cell first { display-mode: "form" }
import importlib, importlib.util, subprocess, sys
for _pkg in ("medmnist",):                           # the only package that Colab does not have already
    if importlib.util.find_spec(_pkg) is None:
        print(f"installing {_pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
importlib.invalidate_caches()
import io, os, ssl, time, urllib.request, warnings, zipfile
import numpy as np
from scipy import ndimage as ndi
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10, "font.size": 9})

def _ref_rates(y_true, y_pred):                     # reference version, used if the home function does not work yet
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp, tn = np.sum((y_true == 1) & (y_pred == 1)), np.sum((y_true == 0) & (y_pred == 0))
    return (tp + tn) / len(y_true), tp / max(np.sum(y_true == 1), 1), tn / max(np.sum(y_true == 0), 1)
def _ref_iou(pred, gt):                             # reference version of the IoU of Task 2
    union = np.sum(pred | gt)
    return 1.0 if union == 0 else np.sum(pred & gt) / union

def _is_blank(x):
    return x is Ellipsis
print("✅ Setup done")

## The data
PneumoniaMNIST: chest X-rays of children, 64 × 64 pixels (Yang et al., 2023, CC BY 4.0; original images Kermany et al., 2018). Each X-ray is labelled normal or pneumonia. The X-rays come in three parts:
* the **training** set (4,708 X-rays): the model learns from these.
* the **validation** set (524): here we choose settings, such as the threshold.
* the **test** set (624): here we report the result, once.

Run the next cell. It loads the X-rays and shows two normal and two pneumonia X-rays.

In [ ]:
#@title Load the chest X-rays { display-mode: "form" }
from medmnist import PneumoniaMNIST
_train = PneumoniaMNIST(split="train", download=True, size=64, root=".")
_val = PneumoniaMNIST(split="val", download=True, size=64, root=".")
_test = PneumoniaMNIST(split="test", download=True, size=64, root=".")
X_train, y_train = _train.imgs.astype(np.float32) / 255, _train.labels.ravel()
X_val, y_val = _val.imgs.astype(np.float32) / 255, _val.labels.ravel()
X_test, y_test = _test.imgs.astype(np.float32) / 255, _test.labels.ravel()

def flat(X):
    """Keep every second pixel (64 × 64 → 32 × 32) and write each image as one row of 1,024 numbers."""
    return X[:, ::2, ::2].reshape(len(X), -1)

def show_xrays(X, y, n=4, title=""):
    fig, ax = plt.subplots(1, n, figsize=(1.8 * n, 2.0))
    for k, a in enumerate(ax):
        a.imshow(X[k], cmap="gray", vmin=0, vmax=1); a.set_title("pneumonia" if y[k] else "normal"); a.axis("off")
    fig.suptitle(title); plt.show()

for _name, _y in (("Training", y_train), ("Validation", y_val), ("Test", y_test)):
    print(f"{_name + ' images:':19s}{len(_y):5,} ({(_y == 0).sum():,} normal, {(_y == 1).sum():,} pneumonia)")
_k = np.r_[np.where(y_train == 0)[0][:2], np.where(y_train == 1)[0][:2]]
show_xrays(X_train[_k], y_train[_k], title="Two normal and two pneumonia X-rays")

## 1.0 At home: your function
**Recap: TP, FN, FP and TN.** For every child, the model answers "pneumonia" or "normal". We compare this answer with the truth. There are four possible outcomes, and each name is made of two words:
* **positive** or **negative** is the model's answer: positive means "pneumonia", negative means "normal".
* **true** or **false** says if this answer was right.

So a **false negative** is a wrong "normal": a sick child the model missed. The four outcomes form the **confusion matrix** (slide 66):

| | model says **pneumonia** (positive) | model says **normal** (negative) |
|---|---|---|
| **child has pneumonia** | **TP**, true positive: a sick child, found | **FN**, false negative: a sick child, missed |
| **child is healthy** | **FP**, false positive: a healthy child, flagged as sick | **TN**, true negative: a healthy child, correctly called normal |

There are two kinds of error: FN (a sick child is sent home) and FP (a healthy child is treated without need). From the four counts come three rates:
* **accuracy** = (TP + TN) / all children: the share of children who get the right answer.
* **sensitivity** = TP / (TP + FN): of the sick children, the share the model finds.
* **specificity** = TN / (TN + FP): of the healthy children, the share the model correctly calls normal.

**Task.** The next cell contains the function `def rates(y_true, y_pred):`. It is almost complete: only the four lines that count `tp`, `fn`, `fp` and `tn` are missing. The three rates below them are already written.
* **Input:** `y_true` holds the true label of every child, `y_pred` the model's answer for every child. Both are NumPy arrays of the same length, with 0 (normal) or 1 (pneumonia).
* **Your four lines:** each line counts how many children fall into one box of the table above. For example, `tp` is the number of children with `y_true` equal to 1 **and** `y_pred` equal to 1. Each line gives one whole number.
* **Output:** the function returns the three rates: accuracy, sensitivity and specificity.

Then run the cell, and run the check cell below it.

**Example**
```
y_true = np.array([1, 1, 0, 0, 1])
y_pred = np.array([1, 0, 1, 0, 1])
# inside rates: tp = 2, fn = 1, fp = 1, tn = 1
rates(y_true, y_pred)   # about (0.6, 0.667, 0.5)
```
*Explanation:* the first and the last child are sick and found (TP), the second is sick but missed (FN), the third is healthy but flagged (FP), the fourth is healthy and cleared (TN).

<details><summary><b>Hint 1</b></summary>

<code>y_true == 1</code> gives one True or False per child: True for every sick child. Write the same kind of comparison for <code>y_pred</code>, then combine the two.

</details>
<details><summary><b>Hint 2</b></summary>

NumPy joins two True/False arrays element by element with <code>&amp;</code>. The word <code>and</code> does not work on arrays. If you add up True values, you count them.

</details>
<details><summary><b>Hint 3</b></summary>

Do you get an error about an "ambiguous truth value"? Then Python combined the wrong parts: it does <code>&amp;</code> before <code>==</code>. Check your brackets.

</details>

In [ ]:
def rates(y_true, y_pred):
    tp = ...    # YOUR CODE (home): sick, found
    fn = ...    # YOUR CODE (home): sick, missed
    fp = ...    # YOUR CODE (home): healthy, false positive
    tn = ...    # YOUR CODE (home): healthy, cleared
    accuracy = (tp + tn) / len(y_true)
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return accuracy, sensitivity, specificity

In [ ]:
#@title Check rates { display-mode: "form" }
try:
    _ok = np.allclose(rates(np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 1]), np.array([1, 0, 1, 1, 1, 0, 0, 0, 0, 1])), (0.8, 0.8, 0.8)) \
        and np.allclose(rates(np.array([1, 1, 1, 0, 0, 0, 0, 0]), np.array([1, 0, 0, 1, 0, 0, 0, 0])), (0.625, 1 / 3, 0.8))
except Exception:
    _ok = False
if _ok:
    print("✅ rates works. The notebook uses your function.")
else:
    rates = _ref_rates
    print("❌ rates does not work yet: each of tp, fn, fp, tn must be one whole number. Try the example above, then open the hints.\n"
          "   The notebook uses a reference version for now (in class that is fine, just continue).")

## The model (part of 1.0)
The model is a logistic regression: a single neuron, like on slide 46. For each X-ray it gives a **probability** of pneumonia, a number from 0 to 1. If the probability is at least the **threshold**, the answer is "pneumonia"; if not, the answer is "normal". The vendor reported this model at threshold 0.5.

Just run the next cell. It trains the model in a few seconds and prints the vendor's number. In class (1.2) you will move the threshold.

In [ ]:
#@title Train the model (a few seconds) { display-mode: "form" }
model = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
model.fit(flat(X_train), y_train)                    # learns from the training set only
p_val = model.predict_proba(flat(X_val))[:, 1]       # probability of pneumonia, validation set
p_test = model.predict_proba(flat(X_test))[:, 1]     # and test set (used again only in the cell with your thresholds)
acc, sens, spec = rates(y_test, (p_test >= 0.5).astype(int))
print(f"the vendor's number: model at threshold 0.5, test set:  accuracy {acc:.2f}   sensitivity {sens:.2f}   specificity {spec:.2f}")

## 1.1 In class: who labelled the training X-rays?
**The question.** Some labels in the training data are wrong. What does the model learn from them, and what happens on the test set?

**The situation.** A model only learns what its labels tell it (slides 46 and 65). For every training X-ray, a doctor (the "reader") wrote normal or pneumonia. Here the reader was tired and made one kind of mistake: some pneumonia X-rays were labelled **normal**. The test X-rays are labelled correctly.

**Predict first (sheet 1.1, at home):** 20% of the pneumonia X-rays in the training set (20 of every 100) are labelled normal. What happens on the test set?
* **A** the accuracy drops a lot
* **B** the accuracy stays about the same, but the model misses many more sick children
* **C** the accuracy stays about the same, but the model gives more false positives (healthy children flagged as sick)
* **D** nothing changes

**What you do in class**
1. Vote: A, B, C or D (hands up).
2. Find the next cell, the one that starts with `MISSED = 0.2`. Change 0.2 to your group's value: groups 1 and 6: 0.1, groups 2 and 7: 0.2, groups 3 and 8: 0.3, group 4: 0.4, group 5: 0.5. At home, keep 0.2.
3. Run the cell. It takes a few seconds.
4. Read the line `trained on tired reader's labels:`. It gives three numbers: test accuracy, missed sick children and false positives. Compare them with the line `trained on correct labels:` just above it.
5. Say your three numbers. We write them into your column of the table on the slide.

In [ ]:
MISSED = 0.2   # EDIT: the share of pneumonia X-rays that the tired reader labelled normal (0 to 0.5)

# nothing to change below: relabel, train a second model, compare both models on the correct test labels
y_tired = y_train.copy()
_sick = np.where(y_train == 1)[0]
_wrong = np.random.default_rng(0).choice(_sick, int(round(MISSED * len(_sick))), replace=False)
y_tired[_wrong] = 0                                   # these pneumonia X-rays are now labelled normal
model_tired = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000).fit(flat(X_train), y_tired)
print(f"{len(_wrong)} of {len(_sick)} pneumonia X-rays in the training set labelled normal ({MISSED:.0%})\n")
for name, m in (("correct labels", model), ("tired reader's labels", model_tired)):
    y_hat = m.predict(flat(X_test))                   # threshold 0.5
    acc, sens, spec = rates(y_test, y_hat)
    missed = int(((y_test == 1) & (y_hat == 0)).sum()); false_pos = int(((y_test == 0) & (y_hat == 1)).sum())
    print(f"trained on {name + ':':23s} test accuracy {acc:.2f}   missed sick children {missed:3d}   false positives {false_pos:3d}")

## 1.2 In class: where do you put the threshold?
**The question.** The same model is used for two different jobs. Which threshold do you choose for each job?

**The situation.** The model gives every X-ray a number from 0 to 1: how sure it is that the child has pneumonia. You choose a threshold. At or above it, the answer is "pneumonia".
* **Low** threshold: the model says "pneumonia" more often. It finds more sick children (sensitivity goes up), but it also flags more healthy children (specificity goes down).
* **High** threshold: the opposite. Fewer "pneumonia" answers, so specificity goes up and sensitivity goes down.

The two jobs, each with a rule:
* **Scenario A, emergency triage** (deciding fast who needs help first): a missed sick child is the worst error. Rule: sensitivity **0.98** or higher. Take the **highest** threshold that still keeps the rule; it gives the best specificity.
* **Scenario B, before a risky treatment:** a false positive (a healthy child treated) is the worst error. Rule: specificity **0.97** or higher. Take the **lowest** threshold that still keeps the rule; it gives the best sensitivity.

**What you do in class**
1. Run the next cell, **Threshold slider (validation set)**. A slider appears. Click it once, then move it with the arrow keys, in steps of 0.01.
2. Watch the line `validation set, threshold ...`. Sensitivity and specificity change when you move the slider.
3. For A: find the **highest** threshold at which sensitivity is still 0.98 or higher. (Move the slider down until sensitivity is 0.98 or higher. Then move it up one step at a time. When sensitivity falls below 0.98, go one step back.)
4. For B: find the **lowest** threshold at which specificity is still 0.97 or higher. (Move the slider up until specificity is 0.97 or higher. Then move it down one step at a time. When specificity falls below 0.97, go one step back.)
5. In the cell below the slider, write your two thresholds into `threshold_A` and `threshold_B`.
6. Run that cell **once**. It tests your thresholds on the test set: X-rays the model has never seen.
7. Read the two lines `Scenario A, ...` and `Scenario B, ...`: the value on validation, the value on test, and the word near the end of the line, `holds` or `FAILS`: does the rule still hold on test? Write them into the table on sheet 1.2.

In [ ]:
#@title Threshold slider (validation set): move it, read sensitivity and specificity { display-mode: "form" }
def _slider(threshold=0.5):
    acc, sens, spec = rates(y_val, (p_val >= threshold).astype(int))
    print(f"validation set, threshold {threshold:.2f}:  accuracy {acc:.3f}   sensitivity {sens:.3f}   specificity {spec:.3f}")
interact(_slider, threshold=widgets.FloatSlider(value=0.5, min=0.02, max=0.98, step=0.01,
                                                continuous_update=False, description="threshold"));

**The next cell tests your two thresholds on the test set.**
* At home: just run it as it is. Both thresholds are 0.50. This is only a placeholder; the output matters in class.
* In class: write your two thresholds in (steps 5 to 7 above) and run it **once**. Do not change the thresholds after you have seen the test values. That would mean choosing the threshold on the test set.

In [ ]:
threshold_A = 0.50   # EDIT: your threshold for scenario A (emergency triage)
threshold_B = 0.50   # EDIT: your threshold for scenario B (before a risky treatment)

# nothing to change below: this prints the table of sheet 1.2
for name, t, need, target in (("A", threshold_A, 1, 0.98), ("B", threshold_B, 2, 0.97)):
    val = rates(y_val, (p_val >= t).astype(int)); y_t = (p_test >= t).astype(int); test = rates(y_test, y_t)
    rate = "sensitivity" if need == 1 else "specificity"
    print(f"Scenario {name}, threshold {t:.2f}: {rate} {val[need]:.3f} on validation, {test[need]:.3f} on test "
          f"-> requirement {'holds' if test[need] >= target else 'FAILS'} on test")
    print(f"   on test: {int(((y_test == 1) & (y_t == 0)).sum())} of {int((y_test == 1).sum())} sick children missed, "
          f"{int(((y_test == 0) & (y_t == 1)).sum())} of {int((y_test == 0).sum())} healthy children flagged (false positives)")

Does a rule hold on validation but **FAILS** on test? Then you did nothing wrong: this is exactly what the question is about. The test X-rays were collected separately from the validation X-rays, and their healthy children get higher probabilities. Write the result on the sheet as it is. Do not go back to tune the threshold on the test set.

## 1.3 In class: what did this model really learn?
**The question.** A model trained in hospital A works very well there. What did it really learn: the lungs, or something else?

**The situation.** Hospital A X-rays its sickest children in bed with a portable machine. This machine prints a small **"L"** marker in the corner of the image. So in hospital A's data, every pneumonia X-ray has the L, and no normal X-ray has it. Hospital B prints no L. (The L is synthetic: this notebook draws it.)

**What did the model learn?** **A** the lung patterns; **B** the L marker; **C** nothing, it overfitted (it only memorised the training images); **D** the image noise.

**What you do in class**
1. Vote: A, B, C or D (hands up).
2. Run the cell **Hospital A and hospital B**. Read the two lines `hospital A model, tested in hospital A` and `hospital A model, tested in hospital B`. Compare accuracy and sensitivity. Is the accuracy the same in both hospitals?
3. Run the cell **Cover the corner**. It paints the corner with the L black in every test X-ray and counts how many answers of the model change. As a control, it does the same with a corner that has no L. Read both lines: `marker corner covered` and `control (bottom right, no marker) covered`. Say both numbers.
4. Extra, if there is time: run **Extra: where does the model look?** It shows which parts of the X-ray the model uses.

In [ ]:
#@title Hospital A and hospital B { display-mode: "form" }
L_MARKER = np.zeros((7, 5), np.float32); L_MARKER[:, :2] = 1; L_MARKER[-2:, :] = 1    # a small letter "L"

def add_marker(X, which):
    """Copy the images and draw a small white "L" into the top left corner of the images where which is True."""
    X = X.copy()
    X[which, 4:11, 4:9] = np.maximum(X[which, 4:11, 4:9], L_MARKER)
    return X

X_train_A = add_marker(X_train, y_train == 1)   # hospital A: the marker is on every pneumonia image
X_test_A = add_marker(X_test, y_test == 1)
X_test_B = X_test                                # hospital B: no markers at all
_k = [np.where(y_train == 1)[0][0], np.where(y_train == 0)[0][0], np.where(y_train == 1)[0][1], np.where(y_train == 0)[0][1]]
show_xrays(X_train_A[_k], y_train[_k], title="Hospital A")

model_A = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
model_A.fit(flat(X_train_A), y_train)            # trained on hospital A's images
pred_A = model_A.predict(flat(X_test_A))
pred_B = model_A.predict(flat(X_test_B))
for name, pred in (("tested in hospital A", pred_A), ("tested in hospital B", pred_B)):
    acc, sens, spec = rates(y_test, pred)
    print(f"hospital A model, {name}:  accuracy {acc:.2f}   sensitivity {sens:.2f}   specificity {spec:.2f}")

In [ ]:
#@title Cover the corner: does the decision depend on it? { display-mode: "form" }
def count_changed(rows, cols):
    """Cover rows and cols of every hospital A test image with 0 and count the predictions that change."""
    covered = X_test_A.copy(); covered[:, rows, cols] = 0
    return int((model_A.predict(flat(covered)) != pred_A).sum())

print(f"{'marker corner covered:':45s}{count_changed(slice(0, 12), slice(0, 12)):3d} of {len(y_test)} predictions change")
print(f"{'control (bottom right, no marker) covered:':45s}{count_changed(slice(-12, None), slice(-12, None)):3d} of {len(y_test)} predictions change")

In [ ]:
#@title Extra: where does the model look? (cover each part of the image in turn) { display-mode: "form" }
def occlusion_map(model, X, size=8, step=4):
    """Average drop of the pneumonia probability when a grey square covers each part of the images."""
    original = model.predict_proba(flat(X))[:, 1]
    heat, count = np.zeros((64, 64)), np.zeros((64, 64))
    for r in range(0, 64 - size + 1, step):
        for c in range(0, 64 - size + 1, step):
            covered = X.copy(); covered[:, r:r + size, c:c + size] = 0.5
            drop = original - model.predict_proba(flat(covered))[:, 1]
            heat[r:r + size, c:c + size] += drop.mean(); count[r:r + size, c:c + size] += 1
    return heat / count

sick = y_test == 1
heat_A = occlusion_map(model_A, X_test_A[sick][:100])
heat_1 = occlusion_map(model, X_test[sick][:100])
fig, ax = plt.subplots(1, 2, figsize=(7, 3.6))
for a, img, heat, title in ((ax[0], X_test_A[sick][0], heat_A, "hospital A model"), (ax[1], X_test[sick][0], heat_1, "model of 1.2")):
    a.imshow(img, cmap="gray"); a.imshow(heat, cmap="hot", alpha=0.55, vmin=0, vmax=max(heat_A.max(), heat_1.max()))
    a.set_title(f"{title}: where it looks"); a.axis("off")
plt.show()
print("A grey square was slid over each image; the brighter a spot, the more the pneumonia probability dropped when that spot was covered.")

The model took a shortcut: it used the marker instead of the lungs. This is called **shortcut learning**.

**Discussion (sheet 1.3):** your hospital wants to use a model that was trained in another hospital. What would you test first? Name three checks.

---
# Task 2: Image segmentation, which cases fail, and how?
*Lecture slides 70 to 77.*

**What is a colonoscopy?** A doctor moves a thin, flexible tube with a small camera at its tip into the bowel (the large intestine). The camera shows the inner wall of the bowel on a screen. The doctor looks for polyps and removes them during the same examination.

**What is a polyp?** A polyp is a small growth on the inner wall of the bowel. Some polyps turn into cancer, so they are removed.

**The situation.** During a colonoscopy, a camera looks inside the bowel, and the doctor removes polyps. A network that draws the outline of every polyp on the screen could guide the doctor, and one day a robot. This is **segmentation**: for every pixel, the network decides "polyp" or "background". Papers report how well the outline of such a network matches the outline drawn by an expert, usually as one mean value.

**What you do in Task 2.** You train a small network and evaluate it the way a paper would (2.0, at home). In class you look at the cases behind the mean: which images fail, and in what way (2.1, 2.2). Then you test what happens when the image gets dark or smoky, as it does in a real operation (2.3).

**The data.** Kvasir-SEG (Jha et al., 2020; for research and education only): 1,000 images from real colonoscopies, each with at least one polyp. For every image, a medical doctor drew the outline of the polyp by hand, and an experienced gastroenterologist (a doctor for the stomach and the bowel) checked it. This outline is the **mask**: it marks every pixel as polyp or not polyp.
* In every picture of this notebook, **green** is this expert mask: the correct answer.
* **Magenta** is the outline that our network predicts.

Here the images are made smaller (80 × 80 pixels), so that the training is fast. The network is a small U-Net (slide 77). It learns from 900 images and is tested on the other 100.

*Runtime:* choose a GPU: *Runtime, Change runtime type, T4 GPU*. Then the training takes about one minute (on a CPU about 12 minutes).

**In class:** start the network early, you need it for 2.2 and 2.3. If the notebook is not running yet, first run the **Setup** cell at the top of the notebook. Then run the two cells below: **Load the colonoscopy images** and **Train the U-Net** (together about 2 minutes with a GPU).

In [ ]:
#@title Load the colonoscopy images (about 1 minute) { display-mode: "form" }
from PIL import Image
KVASIR_URL = "https://datasets.simula.no/downloads/kvasir-seg.zip"     # about 46 MB
SECTIGO_INTERMEDIATE = "http://crt.sectigo.com/SectigoPublicServerAuthenticationCADVR36.crt"

def _ssl_context():
    """Certificate checking stays on. The Kvasir server does not send one intermediate certificate, which breaks
    downloads on Colab; we load it from the address named in the server's own certificate, as browsers do."""
    try:
        import certifi; ctx = ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        ctx = ssl.create_default_context()
    try:
        ctx.load_verify_locations(cadata=urllib.request.urlopen(SECTIGO_INTERMEDIATE, timeout=20).read())
    except Exception:
        pass
    return ctx

if not os.path.exists("kvasir-seg.zip"):
    with urllib.request.urlopen(KVASIR_URL, timeout=60, context=_ssl_context()) as r, open("kvasir-seg.zip.part", "wb") as f:
        while chunk := r.read(1 << 20):
            f.write(chunk)
    os.replace("kvasir-seg.zip.part", "kvasir-seg.zip")

S = 80                                                              # images are resized to 80 × 80 pixels
with zipfile.ZipFile("kvasir-seg.zip") as z:
    _ids = sorted(n.split("/")[-1][:-4] for n in z.namelist() if n.startswith("Kvasir-SEG/masks/") and n.endswith(".jpg"))
    _img = [np.asarray(Image.open(io.BytesIO(z.read(f"Kvasir-SEG/images/{i}.jpg"))).convert("RGB").resize((S, S)), np.float32) / 255 for i in _ids]
    _msk = [np.asarray(Image.open(io.BytesIO(z.read(f"Kvasir-SEG/masks/{i}.jpg"))).convert("L").resize((S, S))) > 127 for i in _ids]
X_seg, Y_seg = np.stack(_img), np.stack(_msk)
_perm = np.random.default_rng(0).permutation(len(X_seg))
X_train_seg, Y_train_seg = X_seg[_perm[100:]], Y_seg[_perm[100:]]   # 900 training images
X_test_seg, Y_test_seg = X_seg[_perm[:100]], Y_seg[_perm[:100]]     # 100 test images

def outline(ax, mask, color):
    if mask.any():
        ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=1.5)

def show_masks(images, masks, titles, preds=None, figsize_per=2.2):
    fig, ax = plt.subplots(1, len(images), figsize=(figsize_per * len(images), figsize_per + 0.4))
    for k, a in enumerate(np.atleast_1d(ax)):
        a.imshow(images[k]); outline(a, masks[k], "lime")
        if preds is not None:
            outline(a, preds[k], "magenta")
        a.set_title(titles[k]); a.axis("off")
    plt.show()

print(f"{len(X_train_seg)} training and {len(X_test_seg)} test images of {S} × {S} pixels")
show_masks(X_test_seg[:4], Y_test_seg[:4], ["expert mask (green)"] * 4)

In [ ]:
#@title Train the U-Net on the 900 training images (about 1 minute on a GPU) { display-mode: "form" }
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"

def conv_block(c_in, c_out):
    """Two 3 × 3 convolutions, each followed by batch normalisation and ReLU."""
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())

class UNet(nn.Module):
    """Encoder (enc1 to enc4) halves the image step by step, the decoder (dec3 to dec1) brings it back to full size,
    and the skip connections (torch.cat) pass the detailed encoder feature maps to the decoder (slide 77)."""
    def __init__(self, c=16):
        super().__init__()
        self.enc1, self.enc2, self.enc3, self.enc4 = conv_block(3, c), conv_block(c, 2 * c), conv_block(2 * c, 4 * c), conv_block(4 * c, 8 * c)
        self.dec3, self.dec2, self.dec1 = conv_block(12 * c, 4 * c), conv_block(6 * c, 2 * c), conv_block(3 * c, c)
        self.out = nn.Conv2d(c, 1, 1)                     # one score per pixel: polyp or not

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        e4 = self.enc4(F.max_pool2d(e3, 2))
        d3 = self.dec3(torch.cat([F.interpolate(e4, scale_factor=2), e3], dim=1))
        d2 = self.dec2(torch.cat([F.interpolate(d3, scale_factor=2), e2], dim=1))
        d1 = self.dec1(torch.cat([F.interpolate(d2, scale_factor=2), e1], dim=1))
        return self.out(d1)

def train_unet(epochs=20, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = UNet().to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=2e-3)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=epochs * (len(X_train_seg) // 16))
    X = torch.tensor(X_train_seg).permute(0, 3, 1, 2); Y = torch.tensor(Y_train_seg, dtype=torch.float32)[:, None]
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        net.train()
        for batch in np.array_split(rng.permutation(len(X)), len(X) // 16):
            x, y = X[batch].to(device), Y[batch].to(device)
            if rng.random() < 0.5: x, y = x.flip(3), y.flip(3)          # mirrored images as extra training data
            if rng.random() < 0.5: x, y = x.flip(2), y.flip(2)
            logits = net(x); p = torch.sigmoid(logits)
            dice = (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)
            loss = F.binary_cross_entropy_with_logits(logits, y) + 1 - dice
            optimizer.zero_grad(); loss.backward(); optimizer.step(); schedule.step()
        if epoch % 5 == 0:
            print(f"epoch {epoch:2d}/{epochs}   loss {loss.item():.3f}   {time.time() - t0:.0f} s")
    return net.eval()

def segment(net, images):
    """Predicted masks (True = polyp) for a stack of images: the network's score per pixel, cut at 0.5."""
    with torch.no_grad():
        x = torch.tensor(np.asarray(images, np.float32)).permute(0, 3, 1, 2).to(device)
        return (torch.sigmoid(net(x)) > 0.5).cpu().numpy()[:, 0]

if device == "cpu":
    print("No GPU: on Colab's CPU the training takes about 12 minutes. At home that is fine; in class choose a GPU "
          "runtime (Runtime, Change runtime type, T4 GPU) before you start, or follow the stress test on the instructor's screen.")
unet = train_unet()
pred_test = segment(unet, X_test_seg)                 # the network's masks for the 100 test images
print("✅ U-Net trained")

## 2.0 At home: how good is the network, and where does it fail?
This is how a segmentation paper evaluates its network, in five steps. Steps 2 and 5 are given. You write the lines marked `YOUR CODE` in steps 1, 3 and 4.

**Step 1: the IoU of one image.** How much do the prediction and the expert mask of **one** image overlap? `pred` and `gt` are the two masks of one image: arrays of 80 × 80 True/False values, True where a pixel is polyp. `pred` is the prediction of the network, `gt` is the ground truth (the expert mask). The **IoU** (intersection over union, slide 74) is: the number of pixels that are polyp in **both** masks, divided by the number of pixels that are polyp in **at least one** mask.

$$\mathrm{IoU} = \frac{|P \cap G|}{|P \cup G|}$$

**IoU in one picture:** lay the expert mask (green) on the prediction (magenta), like two transparent stickers. The intersection is the area covered by both. The union is the area covered by at least one.
* IoU 1.0: the two outlines are identical.
* IoU 0.5: for example, the prediction covers the whole polyp but is twice as large. Or it covers only half of the polyp.
* IoU 0.0: no shared pixel, for example because the network found nothing.

Missed pixels and extra pixels both lower the IoU.

**Task.** Write the function `iou(pred, gt)` in the next cell. Then run it, and run the check cell below it.
* `pred` and `gt` are two masks of the same shape: NumPy arrays of True/False values, True where a pixel is polyp.
* Return the IoU, a number from 0 to 1.
* If neither mask contains a polyp pixel, return `1.0` (nothing to find, nothing found).

**Example 1** (one row of six pixels; your function must work in the same way for the 80 × 80 masks)
```
pred = np.array([1, 1, 1, 0, 0, 0], bool)
gt   = np.array([0, 1, 1, 1, 0, 0], bool)
iou(pred, gt)       # 0.5
```
*Explanation:* 2 pixels are polyp in both masks, 4 pixels in at least one: 2 / 4 = 0.5.

**Example 2:** `pred` equal to `gt` gives `1.0`. **Example 3:** two masks without a common pixel give `0.0`. **Example 4:** two masks without any polyp pixel give `1.0`.

<details><summary><b>Hint 1</b></summary>

The intersection and the union are both counts of pixels.

</details>
<details><summary><b>Hint 2</b></summary>

In <code>rates</code> you counted the entries that meet two conditions at once. "In at least one mask" is the element-wise <b>or</b> (one result per pixel). NumPy writes it with a single character, like <code>&amp;</code>.

</details>
<details><summary><b>Hint 3</b></summary>

Handle the empty case before you divide.

</details>

In [ ]:
def iou(pred, gt):
    """The IoU of two masks of the same shape (True = polyp): a number from 0 to 1."""
    ...                                # YOUR CODE (home): about 5 lines, see the task above

In [ ]:
#@title Check step 1 { display-mode: "form" }
_a = np.zeros((10, 10), bool); _a[2:6, 2:6] = True          # 16 pixels
_b = np.zeros((10, 10), bool); _b[4:8, 4:8] = True          # 16 pixels, 4 of them overlap with _a
_p, _g = np.array([1, 1, 1, 0, 0, 0], bool), np.array([0, 1, 1, 1, 0, 0], bool)   # example 1
_z = np.zeros((10, 10), bool)                                # no polyp pixel at all
try:
    _ok = np.isclose(iou(_p, _g), 0.5) and np.isclose(iou(_a, _b), 4 / 28) and np.isclose(iou(_a, _a), 1.0) \
        and np.isclose(iou(_a, ~_a), 0.0) and np.isclose(iou(_z, _z), 1.0)
except Exception:
    _ok = False
if _ok:
    iou_fn = iou
    print("✅ iou works: two squares of 16 pixels that share 4 pixels have IoU 4 / 28 = 0.14.")
else:
    iou_fn = _ref_iou
    print("❌ iou does not work yet: the check uses the four examples above and two squares of 16 pixels that share 4.\n"
          "   Try the examples yourself, then open the hints. The notebook uses a reference version for now (in class that is fine, just continue).")

**Steps 2 to 5: all 100 test images.** Now you measure every test image on its own, and then you summarise.
* Step 2 (given): the IoU of each of the 100 test images.
* Steps 3 and 4 (yours): the mean IoU, and the number of images that failed.
* Step 5 (given, in the check cell): show the five worst images, sorted from the worst one up.

**Task.** Write steps 3 and 4 in the next cell, then run it and the check cell below it. `ious` is a NumPy array with the IoU of each of the 100 test images.
* `mean_iou`: one number, the average of `ious`.
* `n_failed`: one whole number, how many images have an IoU below 0.1.

**Example**
```
ious = np.array([0.9, 0.05, 0.7, 0.0, 0.8])
mean_iou            # 0.49
n_failed            # 2
```

<details><summary><b>Hint 1</b></summary>

NumPy arrays have a method for the average. To see all methods of an array, run the cell once. Then type <code>ious.</code> and press Tab.

</details>
<details><summary><b>Hint 2</b></summary>

If you compare the whole array with a number, you get one True or False per image. Count them as in <code>rates</code>.

</details>
<details><summary><b>Hint 3</b></summary>

Python does <code>.sum()</code> before <code>&lt;</code>. So put the comparison in brackets first.

</details>

In [ ]:
# Step 2 (given): the IoU of each of the 100 test images; iou_fn is your iou, or the reference if the check failed
ious = np.array([iou_fn(pred_test[k], Y_test_seg[k]) for k in range(len(Y_test_seg))])

# Step 3: the one number everybody reports
mean_iou = ...                         # YOUR CODE (home): the average of ious

# Step 4: how many images have a very low overlap (IoU below 0.1)?
n_failed = ...                         # YOUR CODE (home): the number of images with an IoU below 0.1

In [ ]:
#@title Check steps 3 and 4, step 5: show the five worst images { display-mode: "form" }
ious = np.array([iou_fn(pred_test[k], Y_test_seg[k]) for k in range(len(Y_test_seg))])   # step 2 again, in case the cell above stopped
# Step 5 (given): the five worst images; np.argsort sorts from the smallest IoU up
worst5 = np.argsort(ious)[:5]
_ref = (float(ious.mean()), int((ious < 0.1).sum()))
try:
    _ok = (not _is_blank(mean_iou)) and (not _is_blank(n_failed)) and np.isclose(float(mean_iou), _ref[0]) and int(n_failed) == _ref[1]
except Exception:
    _ok = False
print("✅ steps 3 and 4 work." if _ok else "❌ steps 3 and 4 are not done yet: the values below are the reference (in class that is fine, just continue).")
print(f"mean IoU over the 100 test images: {_ref[0]:.2f}")
_low = np.where(ious < 0.1)[0]
print(f"images with a very low overlap (IoU below 0.1): {_ref[1]}")
print(f"  of these, the network predicted no polyp at all: {int(sum(pred_test[k].sum() == 0 for k in _low))}")
show_masks(X_test_seg[worst5], Y_test_seg[worst5], [f"image {k}: IoU {ious[k]:.2f}" for k in worst5], preds=pred_test[worst5])
print("The five worst of the 100 test images, sorted: the worst on the left. That is why the IoU grows from left to right.")
print("green = expert mask, magenta = prediction (no magenta outline: the network found nothing)")
print("Sheet 2.2: the mean looks good. What kind of error do the worst images show, and which other kinds of error give a low IoU?")

## 2.1 In class: how good is this prediction?
**The question.** The next cell shows four test images, A, B, C and D (the same four as on the slide). Each has two outlines: green is the expert's, magenta is our network's. How well do they match?

**The situation.** The IoU of the four pictures is **0.29, 0.52, 0.70 and 0.90**, in a mixed order. Which value belongs to which picture? You do not calculate anything: you judge by eye. Remember the IoU picture of 2.0, step 1: the area inside both outlines, divided by the area inside at least one. Missed pixels and extra pixels both lower the IoU.

**What you do in class**
1. Start the network now, if it is not running yet: at the top of Task 2, run **Load the colonoscopy images** and **Train the U-Net** (about 2 minutes on a GPU). You need it for 2.2 and 2.3.
2. Run the next cell, **The four pictures of 2.1**. It needs only the loaded images, not the training.
3. On your own: give each picture one of the four values. Write it on the sheet (2.1).
4. Agree in your group on one answer. Say it.
5. Only then go to the cell **2.1, check your answer**: write your group's four values into `our_answer` and run it. It computes the real IoU of each picture with your function `iou` from 2.0 and tells you how many you got right.

In [ ]:
#@title The four pictures of 2.1 (A to D) { display-mode: "form" }
import base64
# The four test images of the slide. The magenta masks were predicted by our U-Net in the reference run and are saved
# here, so that everybody sees the same four pictures as on the slide.
GAME_IMAGES = [24, 12, 56, 50]                       # test image numbers of A, B, C and D
GAME_PREDS = np.unpackbits(np.frombuffer(base64.b64decode(
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAB/gAAAAAAAAAAD/+AAAAAAAAAAP//wAAAAAAAAAP//+AAAAAAAAAH///wAAAAAAAAD///+AAAAAAAAB////wAAAAAAAA////8AAAAAAAAP////gAAAAAAAD////4AAAAAAAB/////AAAAAAAAf////wAAAAAAAH////+AAAAAAAD/////wAAAAAAA/////8AAAAAAAP/////AAAAAAAD/////4AAAAAAA/////+AAAAAAAP/////wAAAAAAD/////+AAAAAAA//////+AAAAAAH//////gAAAAAB//////4AAAAAAf/////+AAAAAAH//////gAAAAAA//////4AAAAAAP/////+AAAAAAB//////AAAAAAAf/////AAAAAAAD/////wAAAAAAAf////8AAAAAAAD/////AAAAAAAA/////wAAAAAAAH////8AAAAAAAAP////AAAAAAAAB////gAAAAAAAAP///wAAAAAAAAB///4AAAAAAAAAP//8AAAAAAAAAD//+AAAAAAAAAA///gAAAAAAAAAAP/wAAAAAAAAAAA/4AAAAAAAAAAAD8AAAAAAAAAAAAeAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAHwAAAAAAAAAAAD+AAAAAAAAAAAA/wAAAAAAAAAAA/+AAAAAAAAAAAf/wAAAAAAAAAAH/+AAAAAAAAAAD//gAAAAAAAAAB//8AAAAAAAAAB///AAAAAAAAAAf//wAAAAAAAAAD//8AAAAAAAAAA///gAAAAAAAAAH//4AAAAAAAAAB//+AAAAAAAAAAP//gAAAAAAAAAD//wAAAAAAAAAAf/8AAAAAAAAAAD//AAAAAAAAAAAf/gAAAAAAAAAAD/wAAAAAAAAAAAPwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAP4AAAAAAAAAAAf/AAAAAAAAAAAP/4AAAAAAAAAAH//AAAAAAAAAAD//8AAAAAAAAAA///gAAAAAAAAAf//8AAAAAAAAAH///AAAAAAAAAD///4AAAAAAAAB////AAAAAAAAA////wAAAAAAAAP///8AAAAAAAAD////gAAAAAAAB////4AAAAAAAAf///+AAAAAAAAH////gAAAAAAAB////4AAAAAAAAf///+AAAAAAAAD////gAAAAAAAA////4AAAAAAAAH///+AAAAAAAAA////AAAAAAAAAP///wAAAAAAAAB///8AAAAAAAAAP///AAAAAAAAAD///wAAAAAAAAA///8AAAAAAAAAP///AAAAAAAAAD///wAAAAAAAAA///8AAAAAAAAAP//+AAAAAAAAAD///gAAAAAAAAA///4AAAAAAAAAP//+AAAAAAAAAB///gAAAAAAAAAf//4AAAAAAAAAD///AAAAAAAAAA///wAAAAAAAAAP//+AAAAAAAAAD///gAAAAAAAAAf//4AAAAAAAAAH//+AAAAAAAAAA///gAAAAAAAAAP//4AAAAAAAAAD//+AAAAAAAAAAf//AAAAAAAAAAH//wAAAAAAAAAB//4AAAAAAAAAAP/8AAAAAAAAAAB/8AAAAAAAAAAAP8AAAAAAAAAAAB/AAAAAAAAAAAAPAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAB4AAAAAAAAAAAf/AAAAAAAAAAAf/4AAAAAAAAAAP/+AAAAAAAAAAA//wAAAAAAAAAAB/8AAAAAAAAAAAf/AAAAAAAAAAAP/4AAAAAAAAAAD/+AAAAAAAAAAA//gAAAAAAAAAAD/4AAAAAAAAAAAf8AAAAAAAAAAAD/AAAAAAAAAAAAfwAAAAAAAAAAAB4AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAYAAAAAAAAAAAA/AAAAAAAAAAAAf4AAAAAAAAAAA//AAAAAAAAAAAf/wAAAAAAAAAAP/+AAAAAAAAAAD//gAAAAAAAAAA//4AAAAAAAAAAP/+AAAAAAAAAAD//gAAAAAAAAAA//8AAAAAAAAAAP//AAAAAAAAAAD//wAAAwAAAAAAf/+AAAeAAAAAAH//8AAAAAAAAAB///gAAAAAAAAAP//4AAAAAAAAAA///AAAAAAAAAAA//wAAAAAAAAAAH/8AAAAAAAAAAA//AAAAAAAAAAAP/4AAAAAAAAAAH/+AAAAAAAAAAB//gAAAAAAAAAAf/4AAAAAAAAAAP/+AAAAAAAAAAD//gAAAAAAAAAA//4AAAAAAAAAAP/8AAAAAAAAAAD/+AAAAAAAAAAAf/AAAAAAAAAAAD/gAMAAAAAAAAAPwAHgAAAAAAAAAAAB4AAAAAAAAAAAAcAAAAAAAAAAAAHAAAAAAAAAAAABgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA="), np.uint8))[:4 * S * S].reshape(4, S, S).astype(bool)
show_masks(X_test_seg[GAME_IMAGES], Y_test_seg[GAME_IMAGES], list("ABCD"), preds=GAME_PREDS, figsize_per=2.8)
print("green = expert mask, magenta = prediction of the network")
print("Sheet 2.1: the four IoU values are 0.29, 0.52, 0.70 and 0.90. Which value belongs to which picture?")

In [ ]:
# 2.1, check your answer: write your group's value for each picture (0.29, 0.52, 0.70 or 0.90), then run the cell
our_answer = {"A": None, "B": None, "C": None, "D": None}   # EDIT: your group's four values

# nothing to change below
_f = globals().get("iou_fn", _ref_iou)               # your iou from 2.0, or the reference if the check failed
if any(v is None for v in our_answer.values()):
    print("First write your group's four values into our_answer, then run this cell again.")
else:
    _right = 0
    for _L, _k, _p in zip("ABCD", GAME_IMAGES, GAME_PREDS):
        _real = _f(_p, Y_test_seg[_k]); _ok = abs(our_answer[_L] - _real) < 0.005; _right += _ok
        print(f"{_L}: your value {our_answer[_L]:.2f}, real IoU {_real:.2f}   {'right' if _ok else 'not right'}")
    print(f"{_right} of 4 right. Which one surprised you? Look at the missed pixels and at the extra pixels.")

## 2.2 In class: which cases fail, and how? (output of 2.0)
**The question.** The mean IoU of our network on the 100 test images is good (0.68 on the slide). But a mean can hide images on which the network fails. How many test images have a very low overlap (IoU below 0.1), and what went wrong there?

**What you base your vote on.** At home, in 2.0 step 4, you computed this number yourself (`n_failed`). Vote from memory, without scrolling up. If you skipped the homework, guess: the mean of 0.68 alone does not tell you, and that is the point of this question.

**What you do in class**
1. Vote: how many of the 100 test images have an IoU below 0.1? **A** none; **B** 1 or 2; **C** 3 to 10; **D** more than 10 (hands up).
2. Scroll up to the end of 2.0, to the cell **Check steps 3 and 4, step 5: show the five worst images**. You do not need to run anything new. If you skipped the homework: run *Runtime, Run all* first. The check cell then shows the reference values.
3. Read the line `images with a very low overlap (IoU below 0.1)`, and the line below it, `of these, the network predicted no polyp at all`.
4. Look at the five worst images below these lines. Green is the expert's outline, magenta the network's. No magenta outline means: the network found nothing. What kind of error do you see? Write it on the sheet (2.2, question 1).
5. Which **other** kinds of error also give a low IoU (sheet, question 2)? Think of the IoU picture in 2.0, step 1, and of pictures A and B in 2.1: what else makes the overlap small?
6. What should a paper report besides the mean (sheet, question 3)? Think of the numbers this output prints.

Your mean may differ from the slide by a few hundredths, and the number of failed images by a few, because every training run is slightly different.

## 2.3 In class: stress test
**The question.** In the operating room, images are not as clean as in our test set: smoke, reflections, blur, instruments in the way, poor light. Does the network still find the polyp?

**The idea.** So far we tested the network on clean images: mean IoU 0.68 (2.0). Now we make all 100 test images worse on purpose, step by step, and measure the mean IoU again. If it falls a lot, the network is not ready for the operating room.

**The degradations.** Five ways to make an image worse. They are **synthetic**: this notebook makes them, they are not real surgical images (slides 70 and 71). Each has a **strength** from 0 (clean image) to 1 (full strength).

**Your group's degradation:** group 1 smoke, 2 reflection, 3 blur, 4 occlusion (an instrument in the way), 5 underexposure (too dark), 6 smoke, 7 reflection, 8 blur. At home, try any one.

**What you do in class: three steps**
1. **Look.** Run the cell **The five degradations (given)**. It shows one test image, clean and under each degradation at full strength. Find your degradation. Is the polyp still visible?
2. **Predict, before you measure.** Nothing is measured yet. Guess: at full strength, what will the mean IoU of the 100 test images be (clean: 0.68)? Choose one letter:
   * **A** hardly worse: 0.45 or more
   * **B** clearly worse: 0.20 to 0.44
   * **C** the network fails almost completely: below 0.20

   Write your letter on the sheet (2.3, column "Prediction") and on the board.
3. **Measure.** In the cell **2.3, step 3**, write your degradation between the quotes in the line `degradation = ""` (for example `degradation = "blur"`) and run it. The cell below it explains each line of the output. Write the measured value on the sheet (column "Measured"), and answer the last question: does your degradation mainly hide the polyp, or change how it looks?

In [ ]:
#@title The five degradations (given) { display-mode: "form" }
def smoke(images, strength):
    """A white veil whose density varies over the image; the pattern of image k is seeded with k."""
    out = np.empty_like(images)
    for k, x in enumerate(images):
        haze = ndi.gaussian_filter(np.random.default_rng(k).random(x.shape[:2]), 10)
        haze = (haze - haze.min()) / np.ptp(haze); a = 0.7 * strength * haze[..., None]
        out[k] = np.clip(x * (1 - a) + a, 0, 1)
    return out

def reflection(images, strength):
    """Up to 8 small pure white discs (specular reflections) per image."""
    out = images.copy(); rr, cc = np.mgrid[:S, :S]
    for k in range(len(images)):
        rng = np.random.default_rng(k)
        for _ in range(int(round(8 * strength))):
            r, c, rad = rng.integers(8, S - 8), rng.integers(8, S - 8), rng.integers(2, 5)
            out[k][(rr - r) ** 2 + (cc - c) ** 2 < rad ** 2] = 1.0
    return out

def blur(images, strength):
    """Gaussian blur with sigma 2.5 * strength pixels (motion or defocus)."""
    return np.stack([ndi.gaussian_filter(x, (2.5 * strength, 2.5 * strength, 0)) for x in images])

def occlusion(images, strength):
    """A dark diagonal band (an instrument shaft) whose width grows with the strength."""
    out = images.copy(); rr, cc = np.mgrid[:S, :S]
    out[:, np.abs((rr - S / 2) - 0.6 * (cc - S / 2)) < 12 * strength] = 0.12
    return out

def underexposure(images, strength):
    """A too dark image: strength 1 keeps a fifth of the brightness."""
    return images * (1 - 0.8 * strength)

DEGRADATIONS = {"smoke": smoke, "reflection": reflection, "blur": blur, "occlusion": occlusion, "underexposure": underexposure}
print("degradations:", ", ".join(DEGRADATIONS))
_k = 83                                               # one test image with a clearly visible polyp
show_masks([X_test_seg[_k]] + [f(X_test_seg[_k:_k + 1], 1.0)[0] for f in DEGRADATIONS.values()], [Y_test_seg[_k]] * 6,
           ["clean"] + list(DEGRADATIONS), figsize_per=1.9)
print("2.3, step 1: one test image, clean and under each degradation at full strength; green = expert mask.")
print("Find your degradation. Is the polyp still visible? Then do step 2: predict A, B or C before you measure.")

### 2.3, step 2: predict first
**Before you run the next cell:** write your letter (A, B or C) on the sheet and on the board. The next cell measures the real value, so a prediction after it is no longer a prediction.

In [ ]:
# 2.3, step 3: measure your degradation
degradation = ""        # EDIT: your group's degradation: smoke, reflection, blur, occlusion or underexposure

# nothing to change below: degrade all 100 test images at six strengths, segment them, and measure the mean IoU
if degradation not in DEGRADATIONS:
    print("Step 3 needs your degradation: write smoke, reflection, blur, occlusion or underexposure between the quotes "
          "in degradation = \"\", then run this cell again. Did you write your prediction first (step 2)?")
else:
    _f = globals().get("iou_fn", _ref_iou)           # your iou from 2.0, or the reference if the check failed
    strengths = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
    curve = []
    for s in strengths:
        preds = segment(unet, DEGRADATIONS[degradation](X_test_seg, s))                 # degrade, then segment
        curve.append(np.mean([_f(preds[k], Y_test_seg[k]) for k in range(len(Y_test_seg))]))
    _v = round(float(curve[-1]), 2)
    _letter = ("A, hardly worse (0.45 or more)" if _v >= 0.45 else "B, clearly worse (0.20 to 0.44)" if _v >= 0.2
               else "C, the network fails almost completely (below 0.20)")
    print(f"{degradation}: mean IoU of the 100 test images at each strength")
    for s, v in zip(strengths, curve):
        print(f"  strength {s:.1f}{' (clean images)' if s == 0 else ' (full strength)' if s == 1 else ''}: {v:.2f}")
    print(f"For sheet 2.3, column Measured: {_v:.2f} at full strength (clean: {curve[0]:.2f}). That is letter {_letter}.")
    print("Was your prediction right?")

    fig, ax = plt.subplots(figsize=(4.6, 3.0))
    ax.plot(strengths, curve, marker="o"); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
    ax.set_xlabel("strength (0 = clean, 1 = full)"); ax.set_ylabel("mean IoU, 100 test images"); ax.set_title(degradation)
    plt.show()
    _clean = [_f(pred_test[k], Y_test_seg[k]) for k in range(len(Y_test_seg))]
    _ex = int(np.argsort(_clean)[::-1][3]); _d = DEGRADATIONS[degradation](X_test_seg, 1.0)[_ex]
    print("Example: a test image that the network segments well when it is clean. Left clean, right at full strength.")
    print("green = expert mask, magenta = prediction (no magenta outline: the network found nothing)")
    show_masks([X_test_seg[_ex], _d], [Y_test_seg[_ex]] * 2, ["clean", f"{degradation}, strength 1"],
               preds=[pred_test[_ex], segment(unet, _d[None])[0]])

### 2.3: how to read the output above
* **The lines `strength 0.0` to `strength 1.0`:** the mean IoU of all 100 test images, from the clean images (0.0, the same value as in 2.0) up to full strength (1.0).
* **The line `For sheet 2.3`:** the value at full strength and its letter. Write the value in the column "Measured" and compare the letter with your prediction.
* **The curve:** the same six numbers as a picture. It shows how fast the network gets worse when the degradation gets stronger.
* **The example image:** one test image, clean (left) and at full strength (right). Look at it for the last question on the sheet: does your degradation mainly **hide** part of the polyp (something covers it), or mainly **change how it looks** (everything is still there, but looks different; lecture slide 71)?
* **Then compare with the other groups** on the board: which degradation hurts most, and is that the one you expected?

---
## Two tasks, two skills
* **Classification: choose the threshold for the use.** Which error costs more depends on what the model is used for. The threshold follows from that. Choose it on validation data, then check it on data from the place where the model will be used. And remember: the labels decide which error a model learns.
* **Segmentation: look at the cases behind the mean.** Which images fail, and how? Nothing found, the wrong place, far too large, or failure when the image gets dark or smoky.